### Fama French Model Decomposition

Paul Grajzl

Friday, November 13
___

In [16]:
import sys
sys.path.append("..")
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from src import data, factors, plotting

# cached data
prices = data.load_prices().dropna(how="all")
returns = data.load_returns().dropna(how="all")
book = data.load_fundamental("book_equity")
shares_all = pd.read_parquet(data.fund_dir / "shares_all.parquet")

# Fama-French exposures
z_size = factors.standardize(factors.size(prices, shares_all))
z_val = factors.standardize(factors.value(book, prices, shares_all))
z_mom = factors.standardize(factors.momentum(prices))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Fama-French Factors: Background

**Three-factor model (Fama and French, 1993):**

$$
r_{i,t} - r_{f,t} = \alpha_i + \beta_i \,(\text{Mkt} - r_f)_t + s_i \, \text{SMB}_t + h_i \, \text{HML}_t + \varepsilon_{i,t}
$$

- **Mkt − RF**: market return above the risk-free rate (the CAPM factor)
- **SMB** (Small Minus Big): small stocks minus big stocks, the size effect
- **HML** (High Minus Low): high book-to-market (value) minus low book-to-market (growth), the value effect

Motivation: CAPM couldn't explain why small and value stocks historically earned more than their market betas implied. SMB and HML absorbed most of these anomalies.

**Extensions:**
- **Carhart four-factor (1997)**: adds momentum (**UMD** / **MOM**), built on 12-1 month returns
- **Fama-French five-factor (2015)**: adds **RMW** (robust minus weak profitability) and **CMA** (conservative minus aggressive investment, i.e. low vs high asset growth)

**Construction (portfolio sorts):**
1. Each June, sort all US stocks (NYSE, AMEX, Nasdaq) by size (NYSE median) and book-to-market (30th / 70th percentiles)
2. Form 6 value-weighted portfolios (2 size × 3 value)
3. **SMB** = average of 3 small − average of 3 big; **HML** = average of 2 high-B/M − average of 2 low-B/M
4. Book equity from the prior fiscal year, lagged at least six months (their point-in-time safeguard)

Daily factor returns are published free on Kenneth French's Data Library, back to 1926.

## Classic Fama-French vs Our Barra-Style Version

**The core flip:** Fama-French **observes** factor returns (sorted portfolios) and **estimates** each stock's exposures by time-series regression. Barra-style **knows** each stock's exposures (its characteristics) and **estimates** factor returns by cross-sectional regression each day.

| | Classic Fama-French | Our Barra-style version |
|---|---|---|
| Factor returns | Returns of long-short sorted portfolios | Daily cross-sectional regression slopes |
| Stock exposures | Time-series betas on the factors | Z-scored characteristics (log size, log B/P, 12-1 momentum) |
| Exposure shape | Coarse buckets (median, terciles) | Continuous z-scores: every stock's exact position counts |
| Weighting | Value-weighted portfolios | Equal-weighted OLS (every stock counts the same) |
| Controls | Double sort controls size within value only | Joint regression: each factor is pure, net of all others |
| Rebalancing | Annually (June) | Daily |
| Point-in-time | Fixed 6-month accounting lag | SEC filing dates |
| Value measure | Book equity / December market cap | Log book equity / current market cap |
| Market factor | Value-weighted market minus T-bills | Cross-sectional intercept: return of an average, style-neutral stock |
| Universe | All US stocks, NYSE breakpoints | Today's S&P 500 (survivorship bias) |

**Consequences to keep in mind:**
- Our factor returns are **pure**: e.g. our value return is net of size and momentum, while HML is only size-neutral
- Our results lean toward **large caps** (S&P only) and weight **every stock equally**, so they won't match the published factors exactly; correlation with them is the sanity check
- Profitability (RMW) and investment (CMA) need EDGAR income statement and asset-growth data before the five-factor version is complete

In [7]:
ff_exp = {"Size": z_size, "Value": z_val, "Momentum": z_mom}
res_ff = factors.cs_regression_multi(returns, ff_exp)            # intercept = market

ff = pd.DataFrame({
    "Market": res_ff["intercept"],
    "SMB": -res_ff["Size"],
    "HML": res_ff["Value"],
    "UMD": res_ff["Momentum"],
})

plotting.plot_factor_returns(ff["Market"], name="Market")

In [8]:
plotting.plot_factor_compare(
    {c: ff[c] for c in ["SMB", "HML", "UMD"]},
    title="Barra-style Fama-French factors (S&P 500)",
)

In [9]:
plotting.plot_table(
    factors.factor_stats(ff), title="Barra-style Fama-French + momentum: factor returns",
    index_name="Factor", fmt={"Days": "{:,.0f}"},
)

In [12]:
ff_off = data.get_ff_factors()
ff_off["Mkt"] = ff_off["Mkt-RF"] + ff_off["RF"]          # total market return, to match ours
ff_off = ff_off.add_prefix("FF_")

pairs = {"Market": "FF_Mkt", "SMB": "FF_SMB", "HML": "FF_HML", "UMD": "FF_Mom"}
both = ff.join(ff_off, how="inner")

corr = pd.Series({ours: both[ours].corr(both[off]) for ours, off in pairs.items()},
                 name="Correlation with official")
plotting.plot_table(corr, title="Our Barra-style factors vs official Fama-French",
                    subtitle=f"{both.index.min():%b %Y} to {both.index.max():%b %Y}, daily",
                    index_name="Factor")

In [14]:
plotting.plot_factor_compare(
    {"SMB (ours)": both["SMB"], "SMB (official)": both["FF_SMB"]},
    title="SMB: ours vs official",
)

### Barra-Style Fama-French: Results and Validation

**Model:** daily cross-sectional regression of S&P 500 returns on z-scored size, value and momentum exposures, no sectors. The intercept is the market factor.

$$
r_{i,t} = f^{\text{mkt}}_t + f^{\text{size}}_t \, z^{\text{size}}_{i} + f^{\text{value}}_t \, z^{\text{value}}_{i} + f^{\text{mom}}_t \, z^{\text{mom}}_{i} + \varepsilon_{i,t}
$$

Mapped to Fama-French conventions: **SMB = −size** (small minus big), **HML = value**, **UMD = momentum**.

**Factor returns (~4 years, 1,000 days):**

| Factor | Ann. return | Ann. vol | Sharpe | t-stat |
|---|---|---|---|---|
| Market | 17.9% | 14.7% | 1.22 | 2.43 |
| SMB | 4.3% | 3.1% | 1.38 | 2.75 |
| HML | 2.1% | 3.3% | 0.62 | 1.24 |
| UMD | 7.5% | 8.5% | 0.88 | 1.75 |

**Daily correlation with official Fama-French factors:** Market 0.91, SMB 0.61, HML 0.71, UMD 0.87. Built independently from raw SEC filings and prices, the factors clearly capture the same dimensions.

**Why they differ:**
- **Universe**: S&P 500 only (mid vs mega cap) vs all US stocks (micro vs mega cap). This hits SMB hardest.
- **Purity**: ours are net of each other via the joint regression; HML is only size-neutral.
- **Survivorship**: today's S&P includes small companies that grew into it, biasing our SMB positive even though official SMB has been weak.
- **Correlation ≠ average return**: factors can move together day to day while drifting apart in level.

**Scale: per σ vs portfolio returns.** Our factor returns are regression slopes: the return **per 1σ of exposure**, spread across ~500 large caps with equal weights, hence highly diversified and low-vol. Official factors are returns of **long-short portfolios** of extreme stocks (top vs bottom 30%), across all US stocks including volatile small caps, so they're more volatile. The two live on different scales: **compare correlations and shapes, not magnitudes**.

___

### Replacing S&P with Russell 3000
___

In [21]:
universe = data.get_russell3000()
len(universe), universe["sector"].nunique()

(2565, 11)

In [22]:
prices, volume, failed = data.get_prices_volume(universe["ticker"].tolist())
returns = data.compute_returns(prices)
spy_px, spy_ret = data.get_market("SPY")          # needed for beta later

prices.shape, len(failed), failed[:20]

200/2565
400/2565
600/2565
800/2565
1000/2565
1200/2565
1400/2565
1600/2565
1800/2565
2000/2565
2200/2565
2400/2565
2565/2565


((1255, 2565), 0, [])

### Return Cleaning and the Estimation Universe

### The problem

Expanding from the S&P 500 to the Russell 3000 brings in thousands of small caps, and with them extreme daily returns. The largest absolute daily returns in the raw data were:

- **Data errors**: DEC showed an infinite return (prior price recorded as zero), followed by a +1,900% day. Some other huge one-day moves are likely single bad prints.
- **Real but extreme moves**: small biotechs on trial news, small caps pivoting to crypto treasuries, short squeezes. Moves of +400% to +700% in a day are genuine for these names.

Both are dangerous in a regression. OLS minimizes **squared** residuals, so one +500% day carries the weight of thousands of normal observations and can drag that day's factor returns far more than every other stock combined.

### Estimation universe vs coverage universe

Risk models separate two sets of stocks:

- **Estimation universe**: the clean, reasonably liquid stocks used to **estimate factor returns** each day
- **Coverage universe**: every stock the model is **applied** to (exposures, attribution, risk)

A stock can be excluded from estimation but still be covered: its exposures still exist, and its return can still be decomposed using factor returns estimated from everyone else.

### Cleaning steps

**1. Remove impossible prices.** A price of zero or below is never valid. These are set to missing before computing returns, which removes infinite returns.

**2. Price filter (≥ $5, prior close).** Penny stocks have the noisiest prices and most bad prints, and many extreme movers traded below $5 before their spikes. The filter uses the **prior day's** price, so it never uses information from the day being estimated.

**3. Cap daily returns at ±50% for estimation.** Genuine big movers stay in the regression, but no single stock can dominate a day. Moves beyond ±50% are almost always stock-specific news, i.e. idio, so capping them loses essentially no information about factor returns.

**4. Keep raw returns for attribution.** When decomposing a specific stock, use its **actual** return: factor contributions come from the factor returns, and everything else, including any extreme move, lands in idio.

### Summary

| Step | Applies to | Purpose |
|---|---|---|
| Price ≤ 0 → missing | All data | Remove impossible values |
| Prior price ≥ $5 | Estimation only | Exclude penny stocks |
| Returns capped at ±50% | Estimation only | Stop single stocks dominating |
| Raw returns kept | Attribution | Decompose actual performance |

**Expected size**: roughly 2,000 to 2,300 stocks per day in the estimation universe.

In [24]:
# 1. Data errors: non-positive prices are never valid
prices = prices.where(prices > 0)
returns = data.compute_returns(prices)          # also re-saves the cleaned returns

# 2. Estimation universe: price >= $5 as of the prior close, daily returns capped at ±50%
est = prices.shift(1).reindex_like(returns) >= 5
ret_est = returns.where(est).clip(-0.5, 0.5)

# Checks: stocks per day in the estimation universe, and the new largest moves
print(ret_est.notna().sum(axis=1).describe())
ret_est.stack().abs().sort_values(ascending=False).head(5)

count    1254.000000
mean     2282.538278
std        73.596729
min      2176.000000
25%      2221.000000
50%      2265.000000
75%      2339.000000
max      2426.000000
dtype: float64


Date        Ticker
2025-07-28  CELC      0.5
2026-02-10  NKTR      0.5
2022-02-15  LRMR      0.5
2024-09-09  RLAY      0.5
2022-04-25  NKTX      0.5
dtype: float64

In [ ]:
# 1. Download filings (only new companies; S&P ones are cached)
missing_cik, failed_facts = data.download_all_facts(universe["ticker"].tolist())
len(missing_cik), len(failed_facts)

In [27]:
failed_facts

[('OZK',
  '404 Client Error: Not Found for url: https://data.sec.gov/api/xbrl/companyfacts/CIK0001569650.json'),
 ('PFBC',
  '404 Client Error: Not Found for url: https://data.sec.gov/api/xbrl/companyfacts/CIK0001492165.json')]

In [ ]:
# 2. Build book equity + all three share-count sources in one parse
equity_tags = ["StockholdersEquity",
               "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"]
items = {
    "book_equity": dict(tags=equity_tags),
    "shares": dict(tags=["EntityCommonStockSharesOutstanding"],
                   taxonomy="dei", unit="shares", sum_classes=True),
    "shares_wavg": dict(tags=["WeightedAverageNumberOfSharesOutstandingBasic"], unit="shares"),
    "shares_cso": dict(tags=["CommonStockSharesOutstanding"], unit="shares"),
}
fund = data.build_fundamentals(universe["ticker"].tolist(), items, prices.index)
book = fund["book_equity"]

In [ ]:
# 3. Layer the share counts, zeros -> missing
shares_all = (fund["shares"]
              .combine_first(fund["shares_wavg"])
              .combine_first(fund["shares_cso"]))
shares_all = shares_all.where(shares_all > 0)

cover_missing = fund["shares"].columns[fund["shares"].iloc[-2].isna()].tolist()
still_missing = shares_all.columns[shares_all.iloc[-2].isna()].tolist()

book.iloc[-2].notna().mean(), shares_all.iloc[-2].notna().mean(), len(cover_missing), len(still_missing)

In [ ]:
import yfinance as yf

last_px = prices.ffill().iloc[-1]
mcap = (last_px * shares_all.ffill().iloc[-1]) / 1e9

yh = {}
for i, t in enumerate(cover_missing, 1):
    try:
        yh[t] = yf.Ticker(t).info.get("marketCap")
    except Exception:
        yh[t] = None
    if i % 50 == 0:
        print(f"{i}/{len(cover_missing)}")
yh = pd.Series(yh, dtype=float) / 1e9

cmp = pd.DataFrame({"ours": mcap[cover_missing], "yahoo": yh})
cmp["ratio"] = cmp["ours"] / cmp["yahoo"]

bad = cmp.index[cmp["ours"].isna() | ~cmp["ratio"].between(0.85, 1.15)]
implied = (yh[bad] * 1e9 / last_px[bad]).dropna()
for t, n in implied.items():
    shares_all[t] = n

len(bad), len(implied), shares_all.iloc[-1].notna().mean()

In [ ]:
shares_all = shares_all.reindex_like(prices).bfill().where(prices.notna())
print((prices.notna() & shares_all.isna()).sum(axis=1).describe())

shares_all.to_parquet(data.fund_dir / "shares_all.parquet")

In [32]:
est_x = prices >= 5                                   # estimation universe, on the exposure date
mcap = prices * shares_all

z_size = factors.standardize(factors.size(prices, shares_all), mask=est_x)
z_val = factors.standardize(factors.value(book, prices, shares_all), mask=est_x)
z_mom = factors.standardize(factors.momentum(prices), mask=est_x)

res_ff_r3 = factors.cs_regression_multi(
    ret_est, {"Size": z_size, "Value": z_val, "Momentum": z_mom}, weights=np.sqrt(mcap)
)

ff_r3 = pd.DataFrame({
    "Market": res_ff_r3["intercept"],
    "SMB": -res_ff_r3["Size"],
    "HML": res_ff_r3["Value"],
    "UMD": res_ff_r3["Momentum"],
})

plotting.plot_table(factors.factor_stats(ff_r3), title="Barra-style Fama-French + momentum: Russell 3000",
                    index_name="Factor", fmt={"Days": "{:,.0f}"})

In [33]:
both_r3 = ff_r3.join(ff_off, how="inner")
pairs = {"Market": "FF_Mkt", "SMB": "FF_SMB", "HML": "FF_HML", "UMD": "FF_Mom"}

corr_cmp = pd.DataFrame({
    "S&P 500 (equal-weight OLS)": pd.Series({"Market": 0.911, "SMB": 0.608, "HML": 0.711, "UMD": 0.870}),
    "Russell 3000 (√cap WLS)": pd.Series({o: both_r3[o].corr(both_r3[f]) for o, f in pairs.items()}),
})
plotting.plot_table(corr_cmp, title="Correlation with official Fama-French factors", index_name="Factor")

**Cap-weighted centering.** The intercept in the cross-sectional regression is the return of a stock with **zero exposure to every factor**. With equal-weighted z-scores across ~2,300 Russell stocks, "zero size exposure" means a typical small/mid cap (~$2bn), so the intercept tracked small/mid caps rather than the market (correlation with the official market factor: 0.85).

**Fix:** center each exposure so its **cap-weighted** mean is zero, keeping the **equal-weighted** standard deviation for scale (Barra convention). The cap-weighted market portfolio then has zero exposure to every style, so the intercept becomes approximately the **cap-weighted market return**. Factor slopes are essentially unchanged, since centering only shifts exposures by a constant each day.

In [37]:
z_size = factors.standardize(factors.size(prices, shares_all), mask=est_x, weights=mcap)
z_val = factors.standardize(factors.value(book, prices, shares_all), mask=est_x, weights=mcap)
z_mom = factors.standardize(factors.momentum(prices), mask=est_x, weights=mcap)

res_ff_r3 = factors.cs_regression_multi(
    ret_est, {"Size": z_size, "Value": z_val, "Momentum": z_mom}, weights=np.sqrt(mcap)
)
ff_r3 = pd.DataFrame({"Market": res_ff_r3["intercept"], "SMB": -res_ff_r3["Size"],
                      "HML": res_ff_r3["Value"], "UMD": res_ff_r3["Momentum"]})

both_r3 = ff_r3.join(ff_off, how="inner")
corr_cmp["Russell 3000 (cap-centered)"] = pd.Series({o: both_r3[o].corr(both_r3[f]) for o, f in pairs.items()})
plotting.plot_table(corr_cmp, title="Correlation with official Fama-French factors", index_name="Factor")

**Lesson: center after clipping, never before.**

The first attempt at cap-weighted centering subtracted the cap-weighted mean **before** clipping at ±3σ. Because a few mega-caps dominate the weights, the cap-weighted mean of log size sits far in the right tail, so thousands of small caps landed below −3σ and were all clipped to the same value. That erased the size differences among small caps: SMB's correlation with the official factor collapsed from 0.94 to 0.11, and HML (entangled with size) fell from 0.67 to 0.07. Momentum, roughly symmetric, was barely affected.

**Correct order:**
1. Equal-weighted z-score and clip (symmetric outlier handling)
2. Re-standardize
3. **Shift** by the cap-weighted mean, with **no re-clipping**

A constant shift per day can't change slopes in a regression with an intercept; it only changes what the intercept represents. Check: after the fix, SMB, HML and UMD matched the uncentered version exactly, while Market's correlation with the official factor rose from 0.85 to 0.91.

**General rule:** any nonlinear step (clipping, winsorizing, ranking) should happen on a symmetric, equal-weighted scale. Weighted centering is a final linear shift.

In [39]:
plotting.plot_factor_returns(ff_r3["Market"], name="Market (Russell 3000)")

In [40]:
plotting.plot_factor_compare(
    {c: ff_r3[c] for c in ["SMB", "HML", "UMD"]},
    title="Barra-style Fama-French factors (Russell 3000)",
)

In [41]:
plotting.plot_table(
    factors.factor_stats(ff_r3), title="Barra-style Fama-French + momentum: Russell 3000",
    index_name="Factor", fmt={"Days": "{:,.0f}"},
)

In [43]:
for ours, off in pairs.items():
    o, f = both_r3[ours], both_r3[off]
    scaled = o * (f.std() / o.std())                      # match official vol, shape unchanged
    display(plotting.plot_factor_compare(
        {f"{ours} (ours, vol-matched)": scaled, f"{ours} (official)": f},
        title=f"{ours}: ours vs official Fama-French",
        caption="Our series rescaled to the official factor's volatility; correlation and shape are unaffected.",
    ))

### SMB: S&P 500 vs Russell 3000

| | S&P 500 (equal-weight OLS) | Russell 3000 (√cap WLS, cap-centered) |
|---|---|---|
| Ann. return | +4.3% | **−1.6%** |
| t-stat | 2.75 | −0.77 |
| Correlation with official SMB | 0.61 | **0.94** |

**The sign flipped with the universe.** In the S&P, "small" meant mid caps that had grown into the index, a survivor-biased sample of winners, so SMB looked positive and significant. Across the Russell 3000, small means genuinely small companies, which lagged over this period, matching the official factor's recent weakness.

**Correlation jumped from 0.61 to 0.94**: with true small caps in the universe, our size factor captures essentially the same thing as Fama-French SMB.

**Lesson:** the same factor definition can flip sign depending on the universe. Survivorship bias is strongest for size, since index membership itself is a record of past growth. Today's Russell list still carries some survivorship, but much less than the S&P.